In [ ]:
# import sentencepiece as spm
# sp = spm.SentencePieceProcessor(model_file="tok32k.model")
# texts = open("data/sample.txt", encoding="utf-8").read().split("\n")[:2000]
# n_char = sum(len(t) for t in texts)
# n_tok  = sum(len(sp.encode(t)) for t in texts)
# print(f"bytes/token = {n_char/n_tok:.3f}")            # 压缩率，越大越好
# print(f"unk rate    = {sum(t.count('<unk>') for t in map(sp.encode, texts))/n_tok:.5f}")
# print(sp.encode("样本均值的渐近方差为 σ²/n", out_type=str)[:20])   # 中文与公式切分检查
# print(sp.encode("p-value = 0.0321", out_type=str)[:12])            # 数值是否被切碎

In [ ]:
# def n_params(V, d, L, h, tie_embeddings=False, n_kv_head_ratio=1.0):
#     attn = 2 * d * d + 2 * d * d * n_kv_head_ratio      # q,o 满秩；k,v 按 GQA 缩放
#     ffn  = 3 * d * h                                    # SwiGLU: gate/up/down
#     head = d * V if tie_embeddings else 2 * d * V       # 词嵌入与输出头
#     return head + L * (attn + ffn)

# for name, (V, d, L, h) in {
#     "V=6400,  d=512,  L=8,  h=2048":   (6400, 512, 8, 2048),
#     "V=6400,  d=768,  L=16, h=2048":   (6400, 768, 16, 2048),
#     "V=32000, d=1024, L=12, h=2816":   (32000, 1024, 12, 2816),
# }.items():
#     print(f"{name:28s} {n_params(V,d,L,h)/1e6:8.2f} M")

In [ ]:
# import pandas as pd
# df = pd.read_csv("out/pretrain/train_log.csv")           # 字段: step,loss,lr,gnorm
# df["seg"] = pd.cut(df.step, bins=3, labels=["head", "mid", "tail"])
# print(df.groupby("seg", observed=True).agg(loss_mean=("loss","mean"),   # observed=True 消除 pandas 对分类分组的告警
#                             loss_std=("loss","std"),
#                             gnorm_max=("gnorm","max")))
# # 关注三项：loss 是否单调下降；局部是否出现 spike（相邻步涨幅超过 3 倍标准差）；
# # 梯度范数是否出现数量级跳变（提示数值不稳定）

In [ ]:
# import time

# import torch
# import torch.nn as nn

# torch.manual_seed(20260101)                     # 固定种子：权重初始化与数据都可复现

# # 数据：模拟一个带异方差的回归问题，n=4096，特征 20 维
# n, d = 4096, 20
# g = torch.Generator().manual_seed(20260102)
# X = torch.randn(n, d, generator=g)
# beta = torch.randn(d, generator=g)
# y = X @ beta + 0.5 * X[:, 0] ** 2 * torch.randn(n, generator=g)   # 误差方差随 x0 增大

# # 模型：两层前馈网络（远小于语言模型，但训练循环的部件齐全）
# model = nn.Sequential(nn.Linear(d, 64), nn.GELU(), nn.Linear(64, 1))
# opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
# sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=300, eta_min=3e-4)
# EPOCHS = 300

# t0 = time.time()
# for epoch in range(1, EPOCHS + 1):
#     model.train()
#     opt.zero_grad()
#     loss = ((model(X).squeeze(1) - y) ** 2).mean()      # 前向 + 均方误差
#     loss.backward()                                     # 反向传播
#     gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 梯度裁剪，返回裁剪前范数
#     opt.step()                                          # 参数更新
#     sched.step()                                        # 学习率余弦退火
#     if epoch % 50 == 0 or epoch == 1:
#         print(f"epoch {epoch:4d}  loss {loss.item():.4f}  "
#               f"gnorm {gnorm.item():.3f}  lr {sched.get_last_lr()[0]:.2e}")
# print(f"训练完成，耗时 {time.time() - t0:.1f} s（CPU）")

In [ ]:
# import time, json, urllib.request, statistics
# url = "http://127.0.0.1:8000/v1/chat/completions"
# body = {"model": "minimind-stat",
#         "messages": [{"role": "user", "content": "用一句话说明 HC3 与 HC0 的区别"}],
#         "temperature": 0.2, "max_tokens": 128}
# lat = []
# for _ in range(20):
#     t0 = time.perf_counter()
#     req = urllib.request.Request(url, data=json.dumps(body).encode(),
#                                  headers={"Content-Type": "application/json"})
#     r = json.load(urllib.request.urlopen(req))
#     lat.append(time.perf_counter() - t0)
# print("p50", statistics.median(lat), "p95", sorted(lat)[int(0.95*len(lat))])
# print(r["choices"][0]["message"]["content"][:200])

In [ ]:
# from fastapi import FastAPI
# import hashlib, httpx, time, json, pathlib
# app = FastAPI()
# LOG = pathlib.Path("runs/api_log.jsonl")

# @app.post("/ask")
# async def ask(q: dict):
#     t0 = time.perf_counter()
#     async with httpx.AsyncClient() as c:
#         r = await c.post("http://127.0.0.1:8000/v1/chat/completions",
#                          json={"model": "minimind-stat",
#                                "messages": q["messages"],
#                                "temperature": q.get("temperature", 0.2),
#                                "seed": q.get("seed", 20260101)})
#     rec = {"ts": time.time(), "latency": time.perf_counter()-t0,
#            "prompt_sha": hashlib.sha256(json.dumps(q["messages"]).encode()).hexdigest()[:16],
#            "usage": r.json().get("usage")}
#     LOG.open("a", encoding="utf-8").write(json.dumps(rec) + "\n")
#     return r.json()

In [ ]:
# hours = {"pretrain": 2.0, "mid": 0.5, "sft": 0.6, "dpo": 0.3, "eval": 0.4}
# price_per_hour = 3.0            # 示例单价（元/小时），以实际平台报价为准
# tokens_seen = 1.6e9             # 预训练阶段实际消耗的 token 数，从日志读取
# gpu_cost = sum(hours.values()) * price_per_hour
# print(f"GPU 小时 {sum(hours.values()):.1f} h，费用 {gpu_cost:.1f} 元")
# print(f"每百万 token 成本 {gpu_cost / (tokens_seen/1e6):.3f} 元")

In [ ]:
# from __future__ import annotations

# import argparse
# import hashlib
# import json
# import platform
# import sys
# from pathlib import Path

# import numpy as np
# import pandas as pd
# import statsmodels
# import statsmodels.formula.api as smf
# from statsmodels.stats.diagnostic import het_breuschpagan, het_white
# from statsmodels.stats.outliers_influence import OLSInfluence, variance_inflation_factor
# from statsmodels.stats.stattools import durbin_watson, jarque_bera

# VERSION = "0.3.0"


# def sha256(path: Path) -> str:
#     h = hashlib.sha256()
#     h.update(path.read_bytes())
#     return h.hexdigest()


# def se_table(res, data: pd.DataFrame, cluster: str | None, terms: list[str]) -> dict:
#     """并列输出多种标准误，用于判断结论对方差估计方式的敏感度。"""
#     kinds = ["nonrobust", "HC0", "HC1", "HC2", "HC3"]
#     out = {}
#     for k in kinds:
#         if k == "nonrobust":
#             # 常规标准误直接取 fit() 默认结果：nonrobust 不是 get_robustcov_results
#             # 的合法 cov_type，传进去会直接报错
#             se, pv = res.bse, res.pvalues
#         else:
#             r = res.get_robustcov_results(cov_type=k, use_t=True)
#             se, pv = r.bse, r.pvalues
#         out[k] = {"se": dict(zip(terms, np.asarray(se).ravel().tolist())),
#                   "p": dict(zip(terms, np.asarray(pv).ravel().tolist()))}
#     if cluster is not None:
#         r = res.get_robustcov_results(
#             cov_type="cluster", use_t=True,
#             groups=data[cluster].astype("category").cat.codes.to_numpy())
#         out["cluster"] = {"se": dict(zip(terms, np.asarray(r.bse).ravel().tolist())),
#                           "p": dict(zip(terms, np.asarray(r.pvalues).ravel().tolist())),
#                           "n_groups": int(data[cluster].nunique())}
#     return out


# def main() -> int:
#     ap = argparse.ArgumentParser()
#     ap.add_argument("--data", required=True)
#     ap.add_argument("--formula", required=True)
#     ap.add_argument("--cluster", default=None)
#     ap.add_argument("--alpha", type=float, default=0.05)
#     ap.add_argument("--out", default=None)
#     args = ap.parse_args()

#     path = Path(args.data)
#     if not path.exists():
#         print(f"[error] 数据文件不存在: {path}", file=sys.stderr)
#         return 2
#     df = pd.read_csv(path) if path.suffix == ".csv" else pd.read_parquet(path)

#     n_raw = len(df)
#     cols = [c.strip() for c in args.formula.split("~")[1].split("+") if c.strip()]
#     used = [args.formula.split("~")[0].strip()] + cols + ([args.cluster] if args.cluster else [])
#     missing = [c for c in used if c not in df.columns]
#     if missing:
#         print(f"[error] 数据中缺少变量: {missing}", file=sys.stderr)
#         return 2

#     df = df[used].dropna().reset_index(drop=True)   # 重置索引，避免影响点位置与标签错位
#     n_drop = n_raw - len(df)
#     mod = smf.ols(args.formula, data=df)
#     res = mod.fit()
#     exog_names = list(res.model.exog_names)
#     p = len(exog_names)
#     n = int(res.nobs)
#     if n < p + 2:
#         print(f"[error] 样本量 {n} 不足，需至少 {p + 2}", file=sys.stderr)
#         return 2
#     # 异方差检验
#     bp_lm, bp_p, bp_f, bp_fp = het_breuschpagan(res.resid, res.model.exog)
#     white_lm, white_p, _, _ = het_white(res.resid, res.model.exog)
#     jb, jb_p, skew, kurt = jarque_bera(res.resid)

#     # 共线性：对设计矩阵各列计算 VIF，截距列不报告
#     X = res.model.exog
#     vif = {}
#     for j, name in enumerate(exog_names):
#         if name.lower() in ("intercept", "const"):
#             continue
#         try:
#             vif[name] = float(variance_inflation_factor(X, j))
#         except Exception:
#             vif[name] = None

#     # 影响点
#     infl = OLSInfluence(res)
#     h = np.asarray(infl.hat_matrix_diag)
#     cooks = np.asarray(infl.cooks_distance[0])
#     stud = np.asarray(infl.resid_studentized_external)
#     lev_thr, cook_thr = 2.0 * p / n, 4.0 / n
#     infl_points = [int(i) for i in np.where((h > lev_thr) | (cooks > cook_thr) | (np.abs(stud) > 3))[0]]

#     # 剔除影响点后的系数变化（只报告，不执行剔除）
#     refit = {}
#     if infl_points:
#         r2 = smf.ols(args.formula, data=df.drop(df.index[infl_points])).fit()
#         refit = {"n_dropped": len(infl_points),
#                  "coef_change_pct": {k: float(100 * (r2.params.get(k, np.nan) / res.params.get(k, np.nan) - 1))
#                                      for k in res.params.index if abs(res.params.get(k, 0)) > 1e-12}}

#     recs = []
#     if bp_p < args.alpha:
#         recs.append({"code": "USE_ROBUST_SE",
#                      "text": f"Breusch-Pagan 检验 p = {bp_p:.4g} < {args.alpha}，"
#                              f"建议报告 HC3 稳健标准误（n < 50 时改用 wild bootstrap）。"})
#     if args.cluster is not None:
#         G = int(df[args.cluster].nunique())
#         txt = f"已按 {args.cluster} 聚类，簇数 G = {G}。"
#         if G < 40:
#             txt += " G < 40 时聚类稳健标准误在小样本下可能过度拒绝，建议 wild cluster bootstrap。"
#         recs.append({"code": "CLUSTER_ROBUST", "text": txt})
#     if vif and max(v for v in vif.values() if v is not None) > 10:
#         recs.append({"code": "COLLINEARITY", "text": "存在 VIF > 10 的变量，系数估计对设定敏感。"})
#     if infl_points:
#         recs.append({"code": "INFLUENTIAL", "text": f"检出 {len(infl_points)} 个影响点，剔除后系数变化见 influence.refit。"})
#     if jb_p < args.alpha and n < 50:
#         recs.append({"code": "SMALL_SAMPLE", "text": f"残差非正态（Jarque-Bera p = {jb_p:.4g}）且 n = {n}，"
#                                                     f"依赖渐近正态的推断不可靠。"})
#     caveats = ["本脚本只做诊断，不做模型选择",
#                "统计显著不等于因果效应",
#                "Durbin-Watson 仅作提示，不能替代 HAC 标准误"]

#     payload = {
#         "meta": {"skill": "regression-diagnostics", "version": VERSION,
#                  "python": platform.python_version(), "statsmodels": statsmodels.__version__,
#                  "numpy": np.__version__, "data_sha256": sha256(path)[:16],
#                  "formula": args.formula, "alpha": args.alpha},
#         "sample": {"n_raw": n_raw, "n_used": n, "n_dropped_missing": int(n_drop)},
#         "fit": {"params": {k: float(v) for k, v in res.params.items()},
#                 "bse": dict(zip(exog_names, res.bse.tolist())),
#                 "pvalues": dict(zip(exog_names, res.pvalues.tolist())),
#                 "r2": float(res.rsquared), "n_params": p},
#         "robust_se": se_table(res, df, args.cluster, exog_names),
#         "diagnostics": {"breusch_pagan": {"lm": float(bp_lm), "p_value": float(bp_p)},
#                         "white": {"lm": float(white_lm), "p_value": float(white_p)},
#                         "jarque_bera": {"stat": float(jb), "p_value": float(jb_p),
#                                         "skew": float(skew), "kurtosis": float(kurt)},
#                         "durbin_watson": float(durbin_watson(res.resid)),
#                         "vif": vif},
#         "influence": {"leverage_threshold": lev_thr, "cooks_threshold": cook_thr,
#                       "n_flagged": len(infl_points),
#                       "index_sample": infl_points[:20], "refit": refit},
#         "recommendations": recs,
#         "caveats": caveats,
#     }
#     s = json.dumps(payload, ensure_ascii=False, indent=2)
#     if args.out:
#         Path(args.out).write_text(s, encoding="utf-8")
#     print(s)
#     return 0


# if __name__ == "__main__":
#     raise SystemExit(main())

In [ ]:
# import json
# import subprocess
# import sys
# from pathlib import Path

# import numpy as np
# import pandas as pd

# HERE = Path(__file__).parent
# SCRIPT = HERE.parent / "scripts" / "regression_diagnostics.py"


# def run(csv: Path, formula: str, cluster: str | None = None) -> dict:
#     cmd = [sys.executable, str(SCRIPT), "--data", str(csv), "--formula", formula]
#     if cluster:
#         cmd += ["--cluster", cluster]
#     out = subprocess.run(cmd, capture_output=True, text=True, check=True)
#     return json.loads(out.stdout)


# def case_heteroskedastic():
#     """乘性异方差：BP 检验应当检出。"""
#     rng = np.random.default_rng(20260101)
#     n = 200
#     x1, x2 = rng.normal(size=n), rng.normal(size=n)
#     y = 1.0 + 0.5 * x1 + 0.3 * x2 + np.exp(x1) * rng.normal(size=n)
#     p = HERE / "case_hetero.csv"
#     pd.DataFrame({"y": y, "x1": x1, "x2": x2}).to_csv(p, index=False)
#     d = run(p, "y ~ x1 + x2")
#     assert d["diagnostics"]["breusch_pagan"]["p_value"] < 0.05, "异方差未被检出"
#     assert any(r["code"] == "USE_ROBUST_SE" for r in d["recommendations"])


# def case_collinear():
#     """近共线：VIF 应当超过 10。"""
#     rng = np.random.default_rng(20260102)
#     n = 200
#     x1 = rng.normal(size=n)
#     x2 = x1 + rng.normal(scale=0.01, size=n)      # 与 x1 几乎共线
#     y = 1.0 + 0.5 * x1 + 0.0 * x2 + rng.normal(size=n)
#     p = HERE / "case_collinear.csv"
#     pd.DataFrame({"y": y, "x1": x1, "x2": x2}).to_csv(p, index=False)
#     d = run(p, "y ~ x1 + x2")
#     assert max(v for v in d["diagnostics"]["vif"].values() if v) > 10, "共线性未被检出"
#     assert any(r["code"] == "COLLINEARITY" for r in d["recommendations"])


# def case_clustered():
#     """聚类结构：簇数被正确记录，小簇数触发警告。"""
#     rng = np.random.default_rng(20260103)
#     G, m = 20, 10
#     g = np.repeat(np.arange(G), m)
#     u = rng.normal(scale=1.0, size=G)[g]           # 簇随机效应
#     x = rng.normal(size=G * m)
#     y = 1.0 + 0.4 * x + u + rng.normal(size=G * m)
#     p = HERE / "case_cluster.csv"
#     pd.DataFrame({"y": y, "x": x, "g": g}).to_csv(p, index=False)
#     d = run(p, "y ~ x", cluster="g")
#     assert d["robust_se"]["cluster"]["n_groups"] == G
#     assert any(r["code"] == "CLUSTER_ROBUST" for r in d["recommendations"])


# if __name__ == "__main__":
#     for f in (case_heteroskedastic, case_collinear, case_clustered):
#         f()
#         print("ok:", f.__name__)

In [ ]:
# import json
# import urllib.parse
# import urllib.request
# import xml.etree.ElementTree as ET
# from datetime import datetime, timezone
# from pathlib import Path

# from mcp.server.fastmcp import FastMCP

# ROOT = Path(__file__).parent / "library"
# ROOT.mkdir(exist_ok=True)
# NOTES = ROOT / "notes.jsonl"
# mcp = FastMCP("lit-server")

# ATOM = "{http://www.w3.org/2005/Atom}"


# def _query_arxiv(query: str, max_results: int = 10) -> list[dict]:
#     url = "http://export.arxiv.org/api/query?" + urllib.parse.urlencode({
#         "search_query": f'all:"{query}"',
#         "start": 0, "max_results": max_results,
#         "sortBy": "submittedDate", "sortOrder": "descending"})
#     with urllib.request.urlopen(url, timeout=20) as r:
#         xml = r.read()
#     root = ET.fromstring(xml)
#     out = []
#     for e in root.findall(f"{ATOM}entry"):
#         aid = e.findtext(f"{ATOM}id", "").strip()
#         out.append({
#             "arxiv_id": aid.rsplit("/abs/", 1)[-1],
#             "title": " ".join(e.findtext(f"{ATOM}title", "").split()),
#             "authors": [a.findtext(f"{ATOM}name", "") for a in e.findall(f"{ATOM}author")],
#             "published": e.findtext(f"{ATOM}published", "")[:10],
#             "summary": " ".join(e.findtext(f"{ATOM}summary", "").split()),
#             "url": aid,
#         })
#     return out


# @mcp.tool()
# def search_arxiv(query: str, max_results: int = 10) -> str:
#     """检索 arXiv 最新论文，返回结构化元数据列表。"""
#     if max_results > 50:
#         max_results = 50
#     papers = _query_arxiv(query, max_results)
#     return json.dumps({"count": len(papers), "papers": papers}, ensure_ascii=False)


# @mcp.tool()
# def get_paper(arxiv_id: str) -> str:
#     """按 arXiv 编号精确获取单篇论文的元数据；编号不存在时返回空结果。"""
#     url = "http://export.arxiv.org/api/query?" + urllib.parse.urlencode(
#         {"id_list": arxiv_id, "max_results": 1})
#     try:
#         with urllib.request.urlopen(url, timeout=20) as r:
#             root = ET.fromstring(r.read())
#     except Exception as exc:                       # 网络失败要显式报告，不许静默
#         return json.dumps({"error": f"arXiv 查询失败: {exc}"}, ensure_ascii=False)
#     entries = root.findall(f"{ATOM}entry")
#     if not entries:
#         return json.dumps({"found": False, "arxiv_id": arxiv_id}, ensure_ascii=False)
#     e = entries[0]
#     return json.dumps({
#         "found": True,
#         "arxiv_id": arxiv_id,
#         "title": " ".join(e.findtext(f"{ATOM}title", "").split()),
#         "authors": [a.findtext(f"{ATOM}name", "") for a in e.findall(f"{ATOM}author")],
#         "published": e.findtext(f"{ATOM}published", "")[:10],
#         "summary": " ".join(e.findtext(f"{ATOM}summary", "").split()),
#         "url": "https://arxiv.org/abs/" + arxiv_id,
#     }, ensure_ascii=False)


# @mcp.tool()
# def to_bibtex(arxiv_id: str, citekey: str = "") -> str:
#     """把 arXiv 元数据转成 BibTeX 条目；元数据取自 arXiv，不凭记忆生成。"""
#     meta = json.loads(get_paper(arxiv_id))
#     if not meta.get("found"):
#         return json.dumps({"error": f"未找到 {arxiv_id}，无法生成 BibTeX"}, ensure_ascii=False)
#     key = citekey or meta["authors"][0].split()[-1].lower() + meta["published"][:4] + \
#           meta["title"].split()[0].lower()
#     authors = " and ".join(meta["authors"])
#     return json.dumps({"citekey": key, "bibtex": (
#         f"@misc{{{key},\n"
#         f"  title  = {{{meta['title']}}},\n"
#         f"  author = {{{authors}}},\n"
#         f"  year   = {{{meta['published'][:4]}}},\n"
#         f"  eprint = {{{arxiv_id}}},\n"
#         f"  url    = {{{meta['url']}}}\n"
#         "}")}, ensure_ascii=False)


# @mcp.tool()
# def save_note(title: str, summary: str, arxiv_id: str = "") -> str:
#     """把一条文献笔记追加到本地 library/notes.jsonl（写操作，需在客户端侧确认）。"""
#     rec = {"ts": datetime.now(timezone.utc).isoformat(timespec="seconds"),
#            "title": title, "summary": summary, "arxiv_id": arxiv_id}
#     with NOTES.open("a", encoding="utf-8") as f:
#         f.write(json.dumps(rec, ensure_ascii=False) + "\n")
#     return json.dumps({"saved": str(NOTES), "title": title}, ensure_ascii=False)


# @mcp.resource("notes://recent")
# def recent_notes() -> str:
#     """返回最近 20 条文献笔记，供 Agent 作为上下文读取。"""
#     if not NOTES.exists():
#         return "[]"
#     lines = NOTES.read_text(encoding="utf-8").strip().split("\n")[-20:]
#     return "[" + ",".join(lines) + "]"


# if __name__ == "__main__":
#     mcp.run(transport="stdio")

In [ ]:
# import asyncio

# from mcp import ClientSession, StdioServerParameters
# from mcp.client.stdio import stdio_client


# async def main() -> None:
#     params = StdioServerParameters(command="python", args=["lit_server.py"])
#     async with stdio_client(params) as (r, w):
#         async with ClientSession(r, w) as s:
#             await s.initialize()
#             tools = await s.list_tools()
#             names = [t.name for t in tools.tools]
#             print("tools:", names)
#             assert {"search_arxiv", "get_paper", "to_bibtex", "save_note"} <= set(names)

#             res = await s.call_tool("search_arxiv",
#                                     {"query": "heteroskedasticity robust standard errors",
#                                      "max_results": 3})
#             payload = res.content[0].text
#             print("search ok, first 200 chars:", payload[:200])

#             res2 = await s.call_tool("to_bibtex", {"arxiv_id": "0000.0000"})
#             print("nonexistent id ->", res2.content[0].text[:120])


# if __name__ == "__main__":
#     asyncio.run(main())

In [ ]:
# import asyncio
# import json

# from mcp import ClientSession, StdioServerParameters
# from mcp.client.stdio import stdio_client

# MAX_RETRY = 2


# async def call_with_retry(session, name: str, args: dict) -> dict:
#     """工具调用的重试与回退：失败重试若干次，仍失败则回退为人工处理标记。"""
#     for k in range(MAX_RETRY + 1):
#         try:
#             res = await session.call_tool(name, args)
#             return json.loads(res.content[0].text)
#         except Exception as exc:
#             if k == MAX_RETRY:
#                 return {"_failed": True, "tool": name, "error": str(exc)}
#             await asyncio.sleep(2 ** k)
#     return {"_failed": True, "tool": name, "error": "unreachable"}


# async def run(topic: str, keep: int = 5) -> list[dict]:
#     params = StdioServerParameters(command="python", args=["lit_server.py"])
#     async with stdio_client(params) as (r, w):
#         async with ClientSession(r, w) as s:
#             await s.initialize()
#             found = await call_with_retry(s, "search_arxiv",
#                                           {"query": topic, "max_results": keep})
#             if found.get("_failed"):
#                 return [found]
#             picked = [p for p in found.get("papers", [])][:keep]
#             for p in picked:
#                 b = await call_with_retry(s, "to_bibtex", {"arxiv_id": p["arxiv_id"]})
#                 p["bibtex"] = b.get("bibtex", "")
#                 p["_bibtex_failed"] = b.get("_failed", False)
#             # 审批节点：写操作前由人确认，此处打印待确认内容并等待输入
#             print("待写入条目：")
#             for p in picked:
#                 print(" -", p["arxiv_id"], "|", p["title"][:60])
#             if input("确认写入文献库? [y/N] ").strip().lower() != "y":
#                 print("已取消写入")
#                 return picked
#             for p in picked:
#                 await call_with_retry(s, "save_note",
#                                       {"title": p["title"],
#                                        "summary": p["summary"][:300],
#                                        "arxiv_id": p["arxiv_id"]})
#             return picked


# if __name__ == "__main__":
#     print(json.dumps(asyncio.run(run("conformal prediction")), ensure_ascii=False)[:500])

In [ ]:
# from __future__ import annotations

# import argparse
# import hashlib
# import json
# import platform
# import time
# from pathlib import Path

# import numpy as np
# from scipy import stats

# J = 1  # 待检验的系数下标：0 为截距，1 为 x1


# def make_X(rng, n: int, design: str) -> np.ndarray:
#     one = np.ones(n)
#     if design == "A":                       # 标准正态设计，杠杆值均匀
#         x1, x2 = rng.normal(size=n), rng.normal(size=n)
#     elif design == "B":                     # 高杠杆设计：x1 对数正态，右尾重
#         x1 = np.exp(rng.normal(size=n)) - np.exp(0.5)
#         x2 = rng.normal(size=n)
#     elif design == "C":                     # 含一个稀疏二元变量
#         x1 = (rng.random(n) < 0.1).astype(float)
#         x2 = rng.normal(size=n)
#     else:
#         raise ValueError(design)
#     return np.column_stack([one, x1, x2])


# def draw_errors(rng, n: int, dist: str) -> np.ndarray:
#     if dist == "normal":
#         z = rng.normal(size=n)
#     elif dist == "t5":
#         z = rng.standard_t(5, size=n) / np.sqrt(5 / 3)   # 标准化到单位方差
#     elif dist == "lognormal":
#         z = (np.exp(rng.normal(size=n)) - np.exp(0.5)) / np.sqrt(np.e ** 2 - np.e)
#     else:
#         raise ValueError(dist)
#     return z


# def fit(X: np.ndarray, Y: np.ndarray):
#     """Y 可以是 n×m 矩阵：单列是常规拟合，多列用于一次算完所有 bootstrap 复制。"""
#     XtX_inv = np.linalg.pinv(X.T @ X)
#     B = XtX_inv @ (X.T @ Y)
#     return B, Y - X @ B, XtX_inv


# def hc_se(X: np.ndarray, E: np.ndarray, XtX_inv: np.ndarray, kind: str, j: int = J) -> np.ndarray:
#     """返回第 j 个系数在多种 HC 形式下的标准误；E 为 n×m，返回长度 m 的向量。"""
#     n, p = X.shape
#     h = np.einsum("ij,jk,ik->i", X, XtX_inv, X)          # 杠杆值 h_ii
#     u = E ** 2
#     if kind == "OLS":
#         s2 = u.sum(axis=0) / (n - p)
#         V = s2 * XtX_inv[j, j]
#         return np.sqrt(np.repeat(V, E.shape[1]))
#     if kind == "HC0":
#         w = np.ones(n)
#     elif kind == "HC1":
#         w = np.full(n, n / (n - p))
#     elif kind == "HC2":
#         w = 1.0 / (1.0 - h)
#     elif kind == "HC3":
#         w = 1.0 / (1.0 - h) ** 2
#     else:
#         raise ValueError(kind)
#     W = u * w[:, None]                                    # n×m
#     M = np.einsum("ia,im,ib->mab", X, W, X)               # m×p×p 肉矩阵
#     V = np.einsum("ac,mcd,db->mab", XtX_inv, M, XtX_inv)  # m×p×p 三明治
#     return np.sqrt(V[:, j, j])


# def wild_bootstrap_p(X, y, rng, B: int = 199, kind: str = "HC3", j: int = J) -> float:
#     """Rademacher wild bootstrap：在原假设下重抽样，统计量与待精化的检验同口径。"""
#     n, p = X.shape
#     XtX_inv = np.linalg.pinv(X.T @ X)
#     b, e, _ = fit(X, y[:, None])
#     keep = [k for k in range(p) if k != j]
#     br, er, _ = fit(X[:, keep], y[:, None])               # 受约束拟合
#     mu = X[:, keep] @ br[:, 0]                            # 原假设下的均值结构
#     t_obs = b[j, 0] / hc_se(X, e, XtX_inv, kind, j)[0]
#     Wg = rng.integers(0, 2, size=(n, B)) * 2 - 1          # Rademacher 权重
#     Bs, Es, _ = fit(X, mu[:, None] + er * Wg)
#     t_star = Bs[j, :] / hc_se(X, Es, XtX_inv, kind, j)
#     return (1 + np.sum(np.abs(t_star) >= abs(t_obs))) / (B + 1)


# def one_rep(rng, n, design, lam, effect, dist, Bboot, kinds):
#     X = make_X(rng, n, design)
#     beta = np.array([1.0, effect, 0.5])
#     sigma = np.sqrt(1.0 + lam * X[:, 1] ** 2)             # 异方差与 x1 绑定
#     y = X @ beta + sigma * draw_errors(rng, n, dist)
#     XtX_inv = np.linalg.pinv(X.T @ X)
#     b, e, _ = fit(X, y[:, None])
#     df = n - X.shape[1]
#     crit = stats.t.ppf(0.975, df)                          # 双侧 5%，t_{n-p} 临界值
#     out = {"beta_hat": float(b[J, 0])}
#     for k in kinds:
#         se = hc_se(X, e, XtX_inv, k, J)[0]
#         out[f"se_{k}"] = float(se)
#         out[f"rej_{k}"] = int(abs(b[J, 0] / se) > crit)
#         out[f"cov_{k}"] = int(abs(b[J, 0] - beta[J]) <= crit * se)
#     out["rej_WB"] = int(wild_bootstrap_p(X, y, rng, Bboot) < 0.05)
#     return out


# def main() -> int:
#     ap = argparse.ArgumentParser()
#     ap.add_argument("--n", type=int, default=50)
#     ap.add_argument("--design", default="A", choices=["A", "B", "C"])
#     ap.add_argument("--lam", type=float, default=1.0, help="异方差强度")
#     ap.add_argument("--dist", default="normal", choices=["normal", "t5", "lognormal"])
#     ap.add_argument("--effect", type=float, default=0.0, help="beta_1 真值；0 用于测水平")
#     ap.add_argument("--R", type=int, default=2000, help="蒙特卡洛重复次数")
#     ap.add_argument("--Bboot", type=int, default=199)
#     ap.add_argument("--seed", type=int, default=20260101)
#     ap.add_argument("--out", default=None)
#     args = ap.parse_args()

#     kinds = ["OLS", "HC0", "HC1", "HC2", "HC3"]
#     children = np.random.SeedSequence(args.seed).spawn(args.R)   # 与并行顺序无关
#     rows, bh, ses = [], [], {k: [] for k in kinds}
#     t0 = time.time()
#     for r in range(args.R):
#         o = one_rep(np.random.default_rng(children[r]), args.n, args.design,
#                     args.lam, args.effect, args.dist, args.Bboot, kinds)
#         rows.append({k: v for k, v in o.items() if not k.startswith(("se_", "beta_"))})
#         bh.append(o["beta_hat"])
#         for k in kinds:
#             ses[k].append(o[f"se_{k}"])
#     bh = np.asarray(bh)

#     def rate(key: str) -> dict:
#         p = float(np.mean([r[key] for r in rows]))
#         mcse = float(np.sqrt(p * (1 - p) / args.R))
#         return {"rate": round(p, 4), "mcse": round(mcse, 4),
#                 "lo": round(p - 1.96 * mcse, 4), "hi": round(p + 1.96 * mcse, 4)}

#     res = {"config": vars(args), "R": args.R,
#            "reject": {k: rate(f"rej_{k}") for k in kinds} | {"WB": rate("rej_WB")},
#            "coverage": {k: rate(f"cov_{k}") for k in kinds},
#            "se_ratio": {k: round(float(np.median(ses[k]) / bh.std(ddof=1)), 4) for k in kinds},
#            "meta": {"python": platform.python_version(), "numpy": np.__version__,
#                     "seed": args.seed, "wall_sec": round(time.time() - t0, 1),
#                     "config_sha": hashlib.sha256(json.dumps(vars(args), sort_keys=True)
#                                                  .encode()).hexdigest()[:16]}}
#     s = json.dumps(res, ensure_ascii=False, indent=2)
#     if args.out:
#         Path(args.out).parent.mkdir(parents=True, exist_ok=True)
#         Path(args.out).write_text(s, encoding="utf-8")
#     print(s)
#     return 0


# if __name__ == "__main__":
#     raise SystemExit(main())

In [ ]:
# import numpy as np, statsmodels.api as sm
# from sim import hc_se
# X = np.column_stack([np.ones(5), np.array([1., 2., 3., 4., 8.])])   # 含一个高杠杆点
# y = np.array([1.2, 2.1, 2.8, 4.2, 7.5])
# r = sm.OLS(y, X).fit()
# print("sm  HC3:", r.get_robustcov_results("HC3").bse)
# XtX_inv = np.linalg.pinv(X.T @ X); b = XtX_inv @ (X.T @ y); e = (y - X @ b)[:, None]
# print("own HC3:", hc_se(X, e, XtX_inv, "HC3", 1))       # 应与上一行一致

# # 2) 边界测试
# print("h_ii 之和:", np.einsum('ij,jk,ik->i', X, XtX_inv, X).sum(), "应等于 p =", X.shape[1])
# print("lam=0 时 sigma 常数:", np.unique(np.sqrt(1 + 0.0 * X[:, 1] ** 2)))


In [ ]:
# import numpy as np
# from scipy import stats

# R, B, J = 1000, 199, 1          # 重复次数、bootstrap 次数、待检验系数下标
# n = 30                          # 小样本 + 高杠杆设计，失真最明显的组合
# rng = np.random.default_rng(20260105)   # 固定种子
# rej = {k: 0 for k in ("OLS", "HC0", "HC3", "WB")}

# for r in range(R):
#     x1 = np.exp(rng.normal(size=n)) - np.exp(0.5)      # 对数正态，右尾重，产生高杠杆点
#     X = np.column_stack([np.ones(n), x1, rng.normal(size=n)])
#     y = X @ np.array([1.0, 0.0, 0.5]) + np.sqrt(1 + 3 * x1 ** 2) * rng.normal(size=n)
#     XtX_inv = np.linalg.pinv(X.T @ X)
#     b = XtX_inv @ (X.T @ y)
#     e = y - X @ b
#     h = np.einsum("ij,jk,ik->i", X, XtX_inv, X)
#     crit = stats.t.ppf(0.975, n - 3)
#     hinv2 = 1.0 / (1.0 - h) ** 2                        # 杠杆值修正因子
#     for k, w in (("OLS", (e @ e) / (n - 3)),
#                  ("HC0", e ** 2),
#                  ("HC3", e ** 2 * hinv2)):
#         se = np.sqrt(w * XtX_inv[J, J]) if k == "OLS" else \
#              np.sqrt((XtX_inv @ (X.T @ (X * w[:, None])) @ XtX_inv)[J, J])
#         rej[k] += int(abs(b[J] / se) > crit)
#     # wild bootstrap：原假设下重抽样，统计量与被精化的检验同为 HC3 口径
#     Xk = X[:, [0, 2]]
#     br = np.linalg.pinv(Xk.T @ Xk) @ (Xk.T @ y)
#     mu = Xk @ br                                        # 原假设下的均值结构
#     er = y - mu
#     Wg = rng.integers(0, 2, size=(n, B)) * 2 - 1        # Rademacher 权重
#     Ys = mu[:, None] + er[:, None] * Wg
#     Bs = XtX_inv @ (X.T @ Ys)
#     Es = Ys - X @ Bs
#     t_star = Bs[J] / np.sqrt((XtX_inv @ np.einsum("ia,im,ib->mab", X, Es**2 * hinv2[:, None], X)
#                               @ XtX_inv)[:, J, J])
#     t_obs = b[J] / np.sqrt((XtX_inv @ (X.T @ (X * (e**2 * hinv2)[:, None])) @ XtX_inv)[J, J])
#     pval = (1 + np.sum(np.abs(t_star) >= abs(t_obs))) / (B + 1)
#     rej["WB"] += int(pval < 0.05)

# print(f"名义水平 0.05，R = {R}，B = {B}，n = {n}，lambda = 3，高杠杆设计")
# for k, v in rej.items():
#     p = v / R
#     print(f"{k}: 拒绝率 {p:.3f}（蒙特卡洛标准误 {np.sqrt(p * (1 - p) / R):.3f}）")

In [ ]:
# import json, glob, pandas as pd
# rows = []
# for f in glob.glob("runs/n*.json"):
#     d = json.loads(open(f, encoding="utf-8").read())
#     c = d["config"]
#     for m, v in d["reject"].items():
#         rows.append({"n": c["n"], "design": c["design"], "lam": c["lam"],
#                      "method": m, "rate": v["rate"], "mcse": v["mcse"],
#                      "lo": v["lo"], "hi": v["hi"], "R": d["R"]})
# df = pd.DataFrame(rows)
# print(df.pivot_table(index=["design", "n", "lam"], columns="method", values="rate")
#         .round(3).to_string())

In [ ]:
# import numpy as np
# rng = np.random.default_rng(20260104)
# for _ in range(200):
#     n = rng.integers(6, 60); p = rng.integers(2, 6)
#     X = np.column_stack([np.ones(n), rng.normal(size=(n, p - 1))])
#     if np.linalg.matrix_rank(X) < p:
#         continue
#     XtX_inv = np.linalg.pinv(X.T @ X)
#     H = X @ XtX_inv @ X.T
#     h = np.diag(H)
#     assert np.allclose(H @ H, H, atol=1e-8)                 # 幂等
#     assert np.allclose(H, H.T, atol=1e-8)                   # 对称
#     assert abs(np.trace(H) - p) < 1e-8                      # 迹等于秩
#     assert (h >= -1e-10).all() and (h <= 1 + 1e-10).all()   # 0 <= h_ii <= 1
#     y = rng.normal(size=n)
#     b = XtX_inv @ (X.T @ y); e = y - X @ b
#     assert np.allclose(X.T @ e, 0, atol=1e-8)               # 残差与列空间正交
# print("属性测试通过：帽子矩阵与残差的代数性质成立")

In [ ]:
# import json, glob, pandas as pd
# rows = []
# for f in sorted(glob.glob("runs/n*.json")):
#     d = json.loads(open(f, encoding="utf-8").read()); c = d["config"]
#     rows.append({"n": c["n"], "design": c["design"], "lam": c["lam"],
#                  **{m: v["rate"] for m, v in d["reject"].items()},
#                  "R": d["R"]})
# df = pd.DataFrame(rows)
# df

In [ ]:
# import numpy as np
# from scipy import stats
# from sim import fit, hc_se, J

# def cluster_case(G=20, m=20, rho=0.5, R=500, seed=7):
#     """G 个簇、每簇 m 个观测，误差含簇随机效应，回归量 x1 取簇级取值。
#     预期：HC3 的拒绝率远高于名义水平——HC 类方法假设观测独立，
#     簇级回归量与簇随机效应的组合使独立假设下的三明治"肉"被系统性低估。"""
#     rng = np.random.default_rng(seed)
#     rej_hc3, rej_ols = [], []
#     for _ in range(R):
#         g = np.repeat(np.arange(G), m)
#         x1 = np.repeat(rng.normal(size=G), m)              # 簇级回归量：同簇取值相同
#         X = np.column_stack([np.ones(G * m), x1])
#         u = rng.normal(scale=np.sqrt(rho), size=G)[g]      # 簇随机效应
#         v = rng.normal(scale=np.sqrt(1 - rho), size=G * m)
#         y = X @ np.array([1.0, 0.0]) + u + v               # H0: beta_1 = 0 成立
#         XtX_inv = np.linalg.pinv(X.T @ X)
#         b, e, _ = fit(X, y[:, None])
#         crit = stats.t.ppf(0.975, G * m - 2)
#         rej_ols.append(abs(b[J, 0] / hc_se(X, e, XtX_inv, "OLS", J)[0]) > crit)
#         rej_hc3.append(abs(b[J, 0] / hc_se(X, e, XtX_inv, "HC3", J)[0]) > crit)
#     return {"OLS": round(float(np.mean(rej_ols)), 3),
#             "HC3": round(float(np.mean(rej_hc3)), 3), "R": R}

# print(cluster_case())